In [ ]:
import torch
import random
import os
import numpy as np
print(torch.cuda.is_available())  # True if GPU is available
print(torch.cuda.device_count())  # Number of GPUs
print(torch.cuda.get_device_name(0))  # GPU model name

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 99
seed_everything(seed);

from google.colab import drive
import sys
drive.mount('/content/drive')
os.chdir("drive/My Drive/Colab Notebooks")
sys.path.append('/content/drive/My Drive/Colab Notebooks/uEDA-Net/')

True
1
NVIDIA A100-SXM4-80GB
Mounted at /content/drive


In [ ]:
!pip install --upgrade --force-reinstall sympy mpmath
!pip install colorednoise torchinfo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 142.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 50.5 MB/s eta 0:00:00
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from data_preprocess import subject_split_and_stack

folder_path = os.path.join(os.getcwd(), 'uEDA-Net')
data_path = os.path.join(folder_path, 'data')

noisy_train, noisy_test, target_train, target_test, independent_test, noise_bank, _ = subject_split_and_stack(dataset_path=data_path,test_ratio=0.2,seed=seed, load_all = True)


📥 Loading dataset: /content/drive/My Drive/Colab Notebooks/uEDA-Net/data/processed_datasets.pt
✔ All datasets loaded successfully!
📥 Loading dataset: /content/drive/My Drive/Colab Notebooks/uEDA-Net/data/validation_set.pt

PUBLIC - TRAIN
----------------------------------------
Segments : 5181
Length   : 512
Subjects : Not available

PUBLIC - VALID
----------------------------------------
Segments : 1403
Length   : 512
Subjects : Not available

CMAD - TRAIN
----------------------------------------
Segments : 58
Length   : 512
Subjects : Not available

CMAD - VALID
----------------------------------------
Segments : 16
Length   : 512
Subjects : Not available

UMAD2 - TRAIN
----------------------------------------
Segments : 404
Length   : 512
Subjects : Not available

UMAD2 - VALID
----------------------------------------
Segments : 128
Length   : 512
Subjects : Not available

DATASET SPLIT SUMMARY
----------------------------------
Train Noisy : (5668, 512)
Train Clean : (5668, 512)
--

In [ ]:
def batch_corrcoef(y_hat, y, eps=1e-8):
    if y_hat.ndim == 3:
        y_hat = y_hat.squeeze(1)
        y = y.squeeze(1)

    # Center
    y_hat_mean = y_hat.mean(dim=-1, keepdim=True)
    y_mean = y.mean(dim=-1, keepdim=True)
    y_hat_centered = y_hat - y_hat_mean
    y_centered = y - y_mean

    # Covariance and standard deviations
    cov = torch.sum(y_hat_centered * y_centered, dim=-1)
    std_y_hat = torch.sqrt(torch.sum(y_hat_centered ** 2, dim=-1) + eps)
    std_y = torch.sqrt(torch.sum(y_centered ** 2, dim=-1) + eps)

    corr = cov / (std_y_hat * std_y + eps)    # (B,)
    return corr.mean()                 # average across batch

def snr_db(noisy, output, clean, eps=1e-8):
    if clean.ndim == 3:
        clean = clean.squeeze(1)
        output = output.squeeze(1)
        noisy = noisy.squeeze(1)

    noise_in_power = torch.sum((noisy - clean) ** 2, dim=-1)
    noise_out_power = torch.sum((output - clean) ** 2, dim=-1)
    ratio = (noise_in_power + eps) / (noise_out_power + eps)
    snr = 10 * torch.log10(ratio)
    return snr.mean().item()

In [ ]:
import torch.nn as nn

class BottleneckAdapter(nn.Module):
    def __init__(self, student_ch=128, teacher_ch=256):
        super().__init__()
        # 1x1 Conv to match channels
        self.project = nn.Conv1d(student_ch, teacher_ch, kernel_size=1)

    def forward(self, x):
        return self.project(x)


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from tqdm import tqdm
from torchinfo import summary
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from model.unet_teacher import Denoising_Youngbin
from model.unet_student import Student_Youngbin
from data_preprocess import noise_augmentation
import torch.nn.functional as F

if __name__ == "__main__":
    X_train = torch.tensor(noisy_train, dtype=torch.float32).unsqueeze(1)
    y_train = torch.tensor(target_train, dtype=torch.float32).unsqueeze(1)
    X_test  = torch.tensor(noisy_test, dtype=torch.float32).unsqueeze(1)
    y_test  = torch.tensor(target_test, dtype=torch.float32).unsqueeze(1)
    X_test2  = torch.tensor(independent_test, dtype=torch.float32).unsqueeze(1)

    print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

    batch_size = 16
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    teacher_model = Denoising_Youngbin().to(device)
    student_model = Student_Youngbin().to(device) # Student_Youngbin
    print(summary(student_model, input_data=[torch.randn(batch_size, 1, 512).to(device), torch.randn(batch_size, 2).to(device)]))

    model_path = os.path.join(os.getcwd(), 'teacher_model_best.pth')
    checkpoint = torch.load(model_path, map_location=device, weights_only=True)
    teacher_model.load_state_dict(checkpoint)
    teacher_model.eval()
    for p in teacher_model.parameters():
        p.requires_grad_(False)

    adapters = nn.ModuleList([
        BottleneckAdapter(student_ch=8,   teacher_ch=16),
        BottleneckAdapter(student_ch=16,  teacher_ch=32),
        BottleneckAdapter(student_ch=32,  teacher_ch=64),
        BottleneckAdapter(student_ch=64,  teacher_ch=128),
        BottleneckAdapter(student_ch=128, teacher_ch=256)
    ]).to(device)

    params_optimizer = list(student_model.parameters()) + list(adapters.parameters())
    optimizer = torch.optim.AdamW(params_optimizer, lr=1e-3, weight_decay=1e-4)
    num_epochs = 200
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    early_stop_patience = 20
    best_val_loss = float('inf')
    patience_counter = 0
    for epoch in range(num_epochs):
        student_model.train()
        adapters.train()
        train_loss = 0.0

        for noisy, clean in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            input_noisy_np = noisy.detach().cpu().squeeze(1).numpy()
            N_aug = input_noisy_np.shape[0]
            doubly_noisy_np = np.zeros_like(input_noisy_np)
            for i in range(N_aug):
              _, sample_noisy,_ = noise_augmentation(
                  input_noisy_np[i:i+1, :],
                  noise_bank,
                  num_epochs=epoch+1
              )
              doubly_noisy_np[i, :] = sample_noisy.squeeze()
            doubly_noisy = torch.tensor(doubly_noisy_np, dtype=torch.float32).unsqueeze(1).to(device)
            clean = clean.to(device)

            mean = doubly_noisy.mean(dim=2, keepdim=True)
            std  = doubly_noisy.std(dim=2, keepdim=True).clamp(min=1e-6)
            clean_norm = (clean - mean) / std
            noisy_norm = (doubly_noisy - mean) / std
            stats_flat = torch.cat([mean.view(-1, 1), std.view(-1, 1)], dim=1)

            recon, zs_1,zs_2,zs_3,zs_4,zs_5 = student_model(noisy_norm,stats_flat, return_bottleneck=True)

            # KD
            with torch.no_grad():
                teacher_recon, zt_1,zt_2,zt_3,zt_4,zt_5 = teacher_model(noisy_norm, stats_flat, return_bottleneck=True)
            student_feats = [zs_1, zs_2, zs_3, zs_4, zs_5]
            teacher_feats = [zt_1, zt_2, zt_3, zt_4, zt_5]
            loss_teacher_feature = 0
            for i in range(5):
                loss_teacher_feature += F.mse_loss(adapters[i](student_feats[i]), teacher_feats[i])
            loss_teacher_output = F.mse_loss(recon, teacher_recon)

            # Loss
            loss_mse = F.mse_loss(recon, clean_norm)
            loss_kd = loss_teacher_output + 0.3*loss_teacher_feature
            loss = 0.5*loss_mse + 0.5*loss_kd

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * noisy.size(0)

        train_loss /= len(train_loader.dataset)
        scheduler.step()

        # Validation
        student_model.eval()
        val_loss_sum = 0.0
        norm_mse_sum = corr_sum = real_mse_sum= snr_sum= real_mae_sum = 0.0
        n_samples = 0

        with torch.no_grad():
            for noisy, clean in test_loader:
                noisy, clean = noisy.to(device), clean.to(device)
                mean = noisy.mean(dim=2, keepdim=True)
                std  = noisy.std(dim=2, keepdim=True).clamp(min=1e-6)
                clean_norm = (clean - mean) / std
                noisy_norm = (noisy - mean) / std
                stats_flat = torch.cat([mean.view(-1, 1), std.view(-1, 1)], dim=1)

                # output
                recon = student_model(noisy_norm,stats_flat)
                loss_mse = F.mse_loss(recon, clean_norm)
                loss = F.l1_loss(recon, clean_norm)

                # 5. Accumulate Metrics
                B = noisy.size(0)
                pred_real = recon * std + mean

                val_loss_sum += loss.item() * B
                real_mse_sum += F.mse_loss(pred_real, clean).item() * B
                norm_mse_sum += loss_mse.item() * B
                real_mae_sum += F.l1_loss(pred_real, clean).item() * B

                corr = batch_corrcoef(pred_real, clean)
                corr_sum += corr * B

                batch_snr = snr_db(noisy, pred_real, clean)
                snr_sum += batch_snr * B

                n_samples += B

        val_loss_avg = val_loss_sum / n_samples
        real_mse_avg = real_mse_sum / n_samples
        norm_mse_avg = norm_mse_sum / n_samples
        avg_snr_imp  = snr_sum / n_samples
        avg_corr = corr_sum / n_samples
        real_mae_avg = real_mae_sum / n_samples

        if val_loss_avg < best_val_loss:
            best_val_loss = val_loss_avg
            torch.save(student_model.state_dict(), "student_model.pth")


        print(f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss_avg:.6f} | MAE: {real_mae_avg:.6f}")
        print(f"MSE: {real_mse_avg:.6f} | corr: {avg_corr:.3f} | SNR_imp: {avg_snr_imp:.2f} dB")
        print("LR:", scheduler.get_last_lr()[0])

    student_model.load_state_dict(torch.load("student_model.pth", map_location=device))
    print("Training complete and model saved.")

X_train: torch.Size([5668, 1, 512]), y_train: torch.Size([5668, 1, 512])
X_test:  torch.Size([1547, 1, 512]), y_test:  torch.Size([1547, 1, 512])
Layer (type:depth-idx)                   Output Shape              Param #
Student_Youngbin                         [16, 1, 512]              --
├─StudentConvBlock: 1-1                  [16, 8, 256]              --
│    └─Sequential: 2-1                   [16, 8, 256]              --
│    │    └─Conv1d: 3-1                  [16, 8, 256]              16
│    │    └─GroupNorm: 3-2               [16, 8, 256]              16
│    └─Conv1d: 2-2                       [16, 1, 512]              6
│    └─Conv1d: 2-3                       [16, 8, 256]              16
│    └─GroupNorm: 2-4                    [16, 8, 256]              16
│    └─FiLM: 2-5                         [16, 8, 256]              --
│    │    └─Linear: 3-3                  [16, 8]                   16
│    │    └─Linear: 3-4                  [16, 8]                   16
│    └─Lea

KeyboardInterrupt: 

In [ ]:
from model.unet_teacher import Denoising_Youngbin
from model.unet_student import Student_Youngbin
from model.Billal_UNet import denoising_model
import torch.nn.functional as F

def to_float(x):
    if torch.is_tensor(x):
        return x.detach().cpu().item()
    return float(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student_model = Student_Youngbin().to(device) # Student_Youngbin Denoising_Youngbin denoising_model
student_model.load_state_dict(torch.load("student_model_best.pth", map_location=device))
model = student_model
model.eval()

mae_list, rmse_list, mse_list = [], [], []
snr_list, corr_list = [], []

with torch.no_grad():
    for i in range(X_test.shape[0]):
        noisy_input = X_test[i:i+1].to(device)
        clean_target = y_test[i:i+1].to(device)

        mean = noisy_input.mean(dim=2, keepdim=True)
        std = noisy_input.std(dim=2, keepdim=True).clamp(min=1e-6)
        noisy_input_norm = (noisy_input - mean) / std
        stats_flat = torch.cat([mean.view(-1, 1), std.view(-1, 1)], dim=1)

        denoised_out_norm = model(noisy_input_norm, stats_flat)
        denoised_out = denoised_out_norm * std + mean

        # losses
        loss_mse = F.mse_loss(denoised_out, clean_target)
        loss_mae = F.l1_loss(denoised_out, clean_target)
        loss_rmse = torch.sqrt(loss_mse)

        mse_list.append(loss_mse.item())
        mae_list.append(loss_mae.item())
        rmse_list.append(loss_rmse.item())

        # SNR & PCC
        snr_list.append(to_float(snr_db(noisy_input, denoised_out, clean_target)))
        corr_list.append(to_float(batch_corrcoef(denoised_out, clean_target)))

# mean ± std
def mean_std(x):
    return np.mean(x), np.std(x)

mae_m, mae_s = mean_std(mae_list)
rmse_m, rmse_s = mean_std(rmse_list)
mse_m, mse_s = mean_std(mse_list)
snr_m, snr_s = mean_std(snr_list)
corr_m, corr_s = mean_std(corr_list)

# print (.3f)
print(f"MAE: {mae_m:.3f} ± {mae_s:.3f} | RMSE: {rmse_m:.3f} ± {rmse_s:.3f}")
print(f"PCC: {corr_m:.3f} ± {corr_s:.3f} | SNR Imp: {snr_m:.3f} ± {snr_s:.3f} dB")


MAE: 0.144 ± 0.240 | RMSE: 0.185 ± 0.285
PCC: 0.890 ± 0.182 | SNR Imp: 12.077 ± 6.423 dB


In [ ]:
import matplotlib.pyplot as plt
model.eval()
idxs = np.random.randint(1,1547,size = 20)

noisy_list = []
denoised_list = []
for idx in idxs:
    with torch.no_grad():
        noisy_input  = X_test[idx:idx+1].to(device)
        clean_target = y_test[idx:idx+1].to(device)
        mean = noisy_input.mean(dim=2, keepdim=True)
        std  = noisy_input.std(dim=2, keepdim=True).clamp(min=1e-3)
        stats_flat = torch.cat([mean.view(-1, 1), std.view(-1, 1)], dim=1)
        noisy_input_norm = (noisy_input - mean) / std
        denoised_out_norm = model(noisy_input_norm,stats_flat)
        denoised_out = denoised_out_norm*std + mean

    noisy_np    = noisy_input.squeeze().cpu().numpy()
    clean_np    = clean_target.squeeze().cpu().numpy()
    denoised_np = denoised_out.squeeze().cpu().numpy()

    plt.figure(figsize=(12, 8))
    plt.subplot(3, 1, 1)
    plt.plot(noisy_np, color='tab:orange')
    plt.title(f"Noisy Input {idx}")
    plt.ylabel("Amplitude")
    plt.grid(alpha=0.3)

    plt.subplot(3, 1, 2)
    plt.plot(denoised_np, color='tab:blue')
    plt.title("Denoised Output (Model Prediction)")
    plt.ylabel("Amplitude")
    plt.grid(alpha=0.3)

    plt.subplot(3, 1, 3)
    plt.plot(clean_np, color='tab:green')
    plt.title("Clean Target (Reference)")
    plt.xlabel("Time (samples)")
    plt.ylabel("Amplitude")
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


Output hidden; open in https://colab.research.google.com to view.